# Silver Layer - Retail Data Engineering Project

## Purpose
###This notebook clean, deduplicate, handle nulls, correct data types, and join datasets.

In [0]:
bronze_base_path = "abfss://retail@saretailsneh.dfs.core.windows.net/bronze"
silver_base_path = "abfss://retail@saretailsneh.dfs.core.windows.net/silver"

tables = [
    "customers",
    "products",
    "stores",
    "transactions"
]

In [0]:
for table in tables:
    df = spark.read.parquet(f"{bronze_base_path}/{table}/")
    
    print(f"{table}: {df.count()} rows")

customers: 5000 rows
products: 500 rows
stores: 50 rows
transactions: 100005 rows


In [0]:
for table in tables:
    print(f"\n{'=' * 50}")
    print(f"SCHEMA: {table.upper()}")
    print(f"{'=' * 50}")
    
    df = spark.read.parquet(f"{bronze_base_path}/{table}/")
    df.printSchema()


SCHEMA: CUSTOMERS
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)


SCHEMA: PRODUCTS
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)


SCHEMA: STORES
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- opening_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)


SCHEMA: TRANSACTIONS
root
 |-- transaction_id: long (nullable = 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
def clean_string_columns(df):
    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    for column in string_columns:
        df = df.withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "",
                None
            ).otherwise(F.trim(F.col(column)))
        )

    return df

###Clean the Customers table

In [0]:
customers_df = spark.read.parquet(
    f"{bronze_base_path}/customers/"
)

customers_silver = (
    clean_string_columns(customers_df)

    # Remove rows without a primary key
    .filter(F.col("customer_id").isNotNull())

    # Remove exact duplicate rows
    .dropDuplicates()

    # Keep the latest record for each customer
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("customer_id")
            .orderBy(F.col("last_modified_date").desc_nulls_last())
        )
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")

    # Standardize email
    .withColumn("email", F.lower(F.col("email")))

    # Add Silver audit timestamp
    .withColumn("silver_processed_at", F.current_timestamp())
)

display(customers_silver.limit(10))

customer_id,customer_name,email,city,state,registration_date,last_modified_date,silver_processed_at
1,Arjun Sharma,customer1@example.com,Delhi,Delhi,2022-09-10T00:00:00.000Z,2022-12-20T00:00:00.000Z,2026-07-06T21:55:26.992Z
2,Ananya Singh,customer2@example.com,Kolkata,West Bengal,2022-02-13T00:00:00.000Z,2022-04-19T00:00:00.000Z,2026-07-06T21:55:26.992Z
3,Ishaan Gupta,customer3@example.com,Hyderabad,Telangana,2024-03-12T00:00:00.000Z,2024-08-15T00:00:00.000Z,2026-07-06T21:55:26.992Z
4,Arjun Jain,customer4@example.com,Bengaluru,Karnataka,2024-05-22T00:00:00.000Z,2024-12-06T00:00:00.000Z,2026-07-06T21:55:26.992Z
5,Arjun Chopra,customer5@example.com,Bengaluru,Karnataka,2024-09-13T00:00:00.000Z,2024-11-30T00:00:00.000Z,2026-07-06T21:55:26.992Z
6,Meera Sharma,customer6@example.com,Jaipur,Rajasthan,2025-04-28T00:00:00.000Z,2025-12-14T00:00:00.000Z,2026-07-06T21:55:26.992Z
7,Aarav Verma,customer7@example.com,Delhi,Delhi,2023-09-28T00:00:00.000Z,2024-04-06T00:00:00.000Z,2026-07-06T21:55:26.992Z
8,Ishaan Singh,customer8@example.com,Hyderabad,Telangana,2024-11-24T00:00:00.000Z,2025-01-12T00:00:00.000Z,2026-07-06T21:55:26.992Z
9,Kiara Chopra,customer9@example.com,Mumbai,Maharashtra,2024-07-03T00:00:00.000Z,2025-04-10T00:00:00.000Z,2026-07-06T21:55:26.992Z
10,Aarav Jain,customer10@example.com,Delhi,Delhi,2022-12-20T00:00:00.000Z,2023-01-02T00:00:00.000Z,2026-07-06T21:55:26.992Z


In [0]:
print("Bronze rows:", customers_df.count())
print("Silver rows:", customers_silver.count())

customers_silver.printSchema()

Bronze rows: 5000
Silver rows: 5000
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    customers_silver.write
    .mode("overwrite")
    .parquet(f"{silver_base_path}/customers/")
)

print("✓ Customers written successfully to Silver")

✓ Customers written successfully to Silver


###Clean the Products table

In [0]:
products_df = spark.read.parquet(
    f"{bronze_base_path}/products/"
)

products_silver = (
    clean_string_columns(products_df)

    # Remove rows without a primary key
    .filter(F.col("product_id").isNotNull())

    # Remove exact duplicate rows
    .dropDuplicates()

    # Keep the latest record for each product
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("product_id")
            .orderBy(F.col("last_modified_date").desc_nulls_last())
        )
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")

    # Remove products with invalid prices
    .filter(
        F.col("unit_price").isNotNull() &
        (F.col("unit_price") > 0)
    )

    # Add Silver audit timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

display(products_silver.limit(10))

product_id,product_name,category,brand,unit_price,last_modified_date,silver_processed_at
1,Prime Product 1,Electronics,Apex,45212.45,2026-06-28T00:00:00.000Z,2026-07-06T22:01:57.916Z
2,Stride Product 2,Electronics,Apex,65954.73,2025-11-01T00:00:00.000Z,2026-07-06T22:01:57.916Z
3,Stride Product 3,Electronics,Nova,17061.05,2025-01-21T00:00:00.000Z,2026-07-06T22:01:57.916Z
4,Apex Product 4,Electronics,Zenith,40023.58,2026-05-05T00:00:00.000Z,2026-07-06T22:01:57.916Z
5,Nova Product 5,Electronics,Zenith,47996.83,2026-06-03T00:00:00.000Z,2026-07-06T22:01:57.916Z
6,Nova Product 6,Electronics,Nexa,66239.55,2025-06-01T00:00:00.000Z,2026-07-06T22:01:57.916Z
7,Orbit Product 7,Electronics,Nova,86805.17,2025-08-07T00:00:00.000Z,2026-07-06T22:01:57.916Z
8,Nova Product 8,Electronics,Prime,76366.56,2025-01-15T00:00:00.000Z,2026-07-06T22:01:57.916Z
9,Nova Product 9,Electronics,Nova,55992.43,2026-03-13T00:00:00.000Z,2026-07-06T22:01:57.916Z
10,Apex Product 10,Electronics,Nexa,52589.11,2026-02-27T00:00:00.000Z,2026-07-06T22:01:57.916Z


In [0]:
print("Bronze rows:", products_df.count())
print("Silver rows:", products_silver.count())

print(
    "Invalid prices in Silver:",
    products_silver
        .filter(
            F.col("unit_price").isNull() |
            (F.col("unit_price") <= 0)
        )
        .count()
)

products_silver.printSchema()

Bronze rows: 500
Silver rows: 500
Invalid prices in Silver: 0
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    products_silver.write
    .mode("overwrite")
    .parquet(f"{silver_base_path}/products/")
)

print("✓ Products written successfully to Silver")

✓ Products written successfully to Silver


###Clean the Stores table

In [0]:
stores_df = spark.read.parquet(
    f"{bronze_base_path}/stores/"
)

stores_silver = (
    clean_string_columns(stores_df)

    # Remove rows without a primary key
    .filter(F.col("store_id").isNotNull())

    # Remove exact duplicate rows
    .dropDuplicates()

    # Keep the latest record for each store
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("store_id")
            .orderBy(F.col("last_modified_date").desc_nulls_last())
        )
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")

    # Remove impossible future opening dates
    .filter(
        F.col("opening_date").isNull() |
        (F.to_date("opening_date") <= F.current_date())
    )

    # Add Silver audit timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

display(stores_silver.limit(10))

store_id,store_name,city,state,region,opening_date,last_modified_date,silver_processed_at
1,RetailHub Chennai 01,Chennai,Tamil Nadu,South,2021-08-11T00:00:00.000Z,2026-06-26T00:00:00.000Z,2026-07-06T22:07:40.870Z
2,RetailHub Pune 02,Pune,Maharashtra,West,2018-09-17T00:00:00.000Z,2026-01-20T00:00:00.000Z,2026-07-06T22:07:40.870Z
3,RetailHub Noida 03,Noida,Uttar Pradesh,North,2023-06-23T00:00:00.000Z,2025-03-22T00:00:00.000Z,2026-07-06T22:07:40.870Z
4,RetailHub Noida 04,Noida,Uttar Pradesh,North,2019-03-23T00:00:00.000Z,2025-08-03T00:00:00.000Z,2026-07-06T22:07:40.870Z
5,RetailHub Jaipur 05,Jaipur,Rajasthan,North,2023-07-18T00:00:00.000Z,2026-05-28T00:00:00.000Z,2026-07-06T22:07:40.870Z
6,RetailHub Kolkata 06,Kolkata,West Bengal,East,2022-06-21T00:00:00.000Z,2025-04-30T00:00:00.000Z,2026-07-06T22:07:40.870Z
7,RetailHub Hyderabad 07,Hyderabad,Telangana,South,2022-09-18T00:00:00.000Z,2025-11-10T00:00:00.000Z,2026-07-06T22:07:40.870Z
8,RetailHub Jaipur 08,Jaipur,Rajasthan,North,2024-04-01T00:00:00.000Z,2025-09-10T00:00:00.000Z,2026-07-06T22:07:40.870Z
9,RetailHub Delhi 09,Delhi,Delhi,North,2022-11-25T00:00:00.000Z,2026-02-19T00:00:00.000Z,2026-07-06T22:07:40.870Z
10,RetailHub Hyderabad 10,Hyderabad,Telangana,South,2018-03-19T00:00:00.000Z,2026-02-15T00:00:00.000Z,2026-07-06T22:07:40.870Z


In [0]:
print("Bronze rows:", stores_df.count())
print("Silver rows:", stores_silver.count())

print(
    "Duplicate store IDs in Silver:",
    stores_silver
        .groupBy("store_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Future opening dates in Silver:",
    stores_silver
        .filter(
            F.to_date("opening_date") > F.current_date()
        )
        .count()
)

stores_silver.printSchema()

Bronze rows: 50
Silver rows: 50
Duplicate store IDs in Silver: 0
Future opening dates in Silver: 0
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- opening_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    stores_silver.write
    .mode("overwrite")
    .parquet(f"{silver_base_path}/stores/")
)

print("✓ Stores written successfully to Silver")

✓ Stores written successfully to Silver


###Clean the Transactions table

In [0]:
transactions_df = spark.read.parquet(
    f"{bronze_base_path}/transactions/"
)

transactions_silver = (
    clean_string_columns(transactions_df)

    # Remove rows without a transaction ID
    .filter(F.col("transaction_id").isNotNull())

    # Remove exact duplicate rows
    .dropDuplicates()

    # Keep the latest record for each transaction
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("transaction_id")
            .orderBy(F.col("last_modified_date").desc_nulls_last())
        )
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")

    # Keep only valid transactions
    .filter(
        F.col("customer_id").isNotNull() &
        F.col("product_id").isNotNull() &
        F.col("store_id").isNotNull() &
        F.col("transaction_date").isNotNull() &
        F.col("quantity").isNotNull() &
        (F.col("quantity") > 0) &
        F.col("unit_price").isNotNull() &
        (F.col("unit_price") > 0)
    )

    # Convert missing discounts to zero
    .withColumn(
        "discount_amount",
        F.coalesce(
            F.col("discount_amount"),
            F.lit(0).cast("decimal(12,2)")
        )
    )

    # Remove invalid discounts
    .filter(F.col("discount_amount") >= 0)

    # Standardize payment method
    .withColumn(
        "payment_method",
        F.upper(F.col("payment_method"))
    )

    # Calculate business metrics
    .withColumn(
        "gross_amount",
        F.round(
            F.col("quantity") * F.col("unit_price"),
            2
        )
    )
    .withColumn(
        "net_amount",
        F.round(
            F.col("gross_amount") - F.col("discount_amount"),
            2
        )
    )

    # Net amount cannot be negative
    .filter(F.col("net_amount") >= 0)

    # Add Silver audit timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

display(transactions_silver.limit(10))

transaction_id,customer_id,product_id,store_id,transaction_date,quantity,unit_price,discount_amount,payment_method,last_modified_date,gross_amount,net_amount,silver_processed_at
1,4743,379,35,2025-06-14T21:59:09.000Z,3,87349.92,0.00,DEBIT CARD,2025-06-16T12:00:37.000Z,262049.76,262049.76,2026-07-06T22:11:21.226Z
2,3700,108,6,2024-11-24T10:25:38.000Z,2,6005.82,0.00,CREDIT CARD,2024-11-26T00:36:22.000Z,12011.64,12011.64,2026-07-06T22:11:21.226Z
3,2333,428,39,2024-05-23T08:22:25.000Z,4,39716.86,0.00,UPI,2024-05-24T05:24:36.000Z,158867.44,158867.44,2026-07-06T22:11:21.226Z
4,2806,350,35,2025-07-10T16:37:45.000Z,3,63330.21,18999.06,NET BANKING,2025-07-12T20:08:30.000Z,189990.63,170991.57,2026-07-06T22:11:21.226Z
5,106,7,39,2025-02-05T16:50:59.000Z,3,86805.17,0.00,UPI,2025-02-05T17:40:32.000Z,260415.51,260415.51,2026-07-06T22:11:21.226Z
6,1054,285,49,2025-01-03T13:41:38.000Z,1,34195.83,0.00,CREDIT CARD,2025-01-06T00:52:31.000Z,34195.83,34195.83,2026-07-06T22:11:21.226Z
7,3816,328,34,2026-05-29T17:45:49.000Z,3,13319.88,3995.96,CREDIT CARD,2026-05-31T01:53:39.000Z,39959.64,35963.68,2026-07-06T22:11:21.226Z
8,894,401,15,2024-02-23T05:38:55.000Z,2,32402.69,3240.27,CREDIT CARD,2024-02-25T01:30:42.000Z,64805.38,61565.11,2026-07-06T22:11:21.226Z
9,3108,386,20,2024-07-15T13:17:48.000Z,1,32469.18,0.00,CREDIT CARD,2024-07-15T21:53:10.000Z,32469.18,32469.18,2026-07-06T22:11:21.226Z
10,2254,238,38,2025-03-24T10:48:15.000Z,1,39714.91,0.00,CASH,2025-03-25T19:31:45.000Z,39714.91,39714.91,2026-07-06T22:11:21.226Z


In [0]:
print("Bronze rows:", transactions_df.count())
print("Silver rows:", transactions_silver.count())

print(
    "Duplicate transaction IDs:",
    transactions_silver
        .groupBy("transaction_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Invalid quantities:",
    transactions_silver
        .filter(F.col("quantity") <= 0)
        .count()
)

print(
    "Invalid prices:",
    transactions_silver
        .filter(F.col("unit_price") <= 0)
        .count()
)

print(
    "Negative net amounts:",
    transactions_silver
        .filter(F.col("net_amount") < 0)
        .count()
)

transactions_silver.printSchema()

Bronze rows: 100005
Silver rows: 100005
Duplicate transaction IDs: 0
Invalid quantities: 0
Invalid prices: 0
Negative net amounts: 0
root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount_amount: decimal(12,2) (nullable = false)
 |-- payment_method: string (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- gross_amount: decimal(24,2) (nullable = true)
 |-- net_amount: decimal(26,2) (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    transactions_silver.write
    .mode("overwrite")
    .parquet(f"{silver_base_path}/transactions/")
)

print("✓ Transactions written successfully to Silver")

✓ Transactions written successfully to Silver


###Reload all Silver tables

In [0]:
customers_final = spark.read.parquet(
    f"{silver_base_path}/customers/"
)

products_final = spark.read.parquet(
    f"{silver_base_path}/products/"
)

stores_final = spark.read.parquet(
    f"{silver_base_path}/stores/"
)

transactions_final = spark.read.parquet(
    f"{silver_base_path}/transactions/"
)

print("✓ All Silver tables loaded successfully")

✓ All Silver tables loaded successfully


In [0]:
silver_summary = [
    ("customers", customers_final.count()),
    ("products", products_final.count()),
    ("stores", stores_final.count()),
    ("transactions", transactions_final.count())
]

summary_df = spark.createDataFrame(
    silver_summary,
    ["table_name", "row_count"]
)

display(summary_df)

table_name,row_count
customers,5000
products,500
stores,50
transactions,100005


###Check referential integrity

In [0]:
orphan_customers = (
    transactions_final
    .join(
        customers_final.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

orphan_products = (
    transactions_final
    .join(
        products_final.select("product_id"),
        on="product_id",
        how="left_anti"
    )
    .count()
)

orphan_stores = (
    transactions_final
    .join(
        stores_final.select("store_id"),
        on="store_id",
        how="left_anti"
    )
    .count()
)

print("REFERENTIAL INTEGRITY CHECK")
print("=" * 40)
print(f"Transactions with invalid customer_id: {orphan_customers}")
print(f"Transactions with invalid product_id:  {orphan_products}")
print(f"Transactions with invalid store_id:    {orphan_stores}")

REFERENTIAL INTEGRITY CHECK
Transactions with invalid customer_id: 0
Transactions with invalid product_id:  0
Transactions with invalid store_id:    0


In [0]:
silver_transactions = spark.read.parquet(
    "abfss://retail@saretailsneh.dfs.core.windows.net/silver/transactions/"
)
print(f"Silver transactions count: {silver_transactions.count():,}")

Silver transactions count: 100,006
